# Walkthrough: is A/P positional fate primed before transcription?

This notebook is the readable path through the analysis. It **imports the package** in
`src/palate_multiome/` rather than redefining anything, so every function shown here is the
one the pipeline runs and the tests cover.

It needs `data/processed/palate_rna.h5ad`, which `scripts/02_build_rna.py` produces. The
outputs below were generated by executing this notebook against that file.

In [1]:
import pandas as pd
import scanpy as sc

from palate_multiome.annotate import detection_fraction
from palate_multiome.config import load_config
from palate_multiome.priming import fate_prediction, remove_axis, stage_axis

cfg = load_config("../config/config.yaml")
adata = sc.read_h5ad("../data/processed/palate_rna.h5ad")
adata

AnnData object with n_obs × n_vars = 38071 × 2500
    obs: 'sample', 'stage', 'nCount_RNA', 'nFeature_RNA', 'percent_mt', 'nCount_ATAC', 'S_score', 'G2M_score', 'phase', 'leiden', 'major', 'celltype'
    var: 'gene_ids', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection', 'mean', 'std'
    uns: 'hvg', 'leiden', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

## 1. Feasibility: the ATAC peak sets share no feature space

Each library was peak-called independently. The table is produced by
`scripts/01_verify_data.py` from the deposited matrices.

In [2]:
verification = pd.read_csv("../results/day1_verification.csv")
sharing = pd.read_csv("../results/peak_id_sharing.csv")
print(f"cells passing count QC: {verification.pass_count_qc.sum():,} "
      f"of {verification.input_cells.sum():,}")
sharing

cells passing count QC: 38,071 of 39,669


,n_libraries_sharing_id,n_peak_ids
0,1,895069
1,2,727
2,3,1


In [3]:
union = sharing.n_peak_ids.sum()
recurring = sharing.loc[sharing.n_libraries_sharing_id > 1, "n_peak_ids"].sum()
print(f"{recurring:,} of {union:,} peak IDs recur in another library "
      f"({100 * recurring / union:.3f}%) -> matrices cannot be concatenated")

728 of 895,797 peak IDs recur in another library (0.081%) -> matrices cannot be concatenated


## 2. RNA baseline: the A/P axis is real, and absent at E12.5

Labels come from `annotate.annotate()` using absolute mean log-normalized expression.
First, a check that the labels are backed by the markers that define them.

In [4]:
obs = adata.obs
raw = adata.raw.to_adata()
markers = ["Shox2", "Msx1", "Meox2", "Tbx22", "Runx2", "Sp7"]
expr = pd.DataFrame(raw[:, markers].X.toarray(), columns=markers, index=obs.index)
expr.groupby(obs.celltype, observed=True).mean().loc[
    ["Anterior", "Posterior", "Osteogenic", "Early/unpatterned mesenchyme"]].round(2)

,Shox2,Msx1,Meox2,Tbx22,Runx2,Sp7
celltype,,,,,,
Anterior,1.65,0.69,0.11,0.08,1.02,0.03
Posterior,0.17,0.05,1.28,0.44,0.41,0.02
Osteogenic,0.78,1.12,0.07,0.01,3.15,0.54
Early/unpatterned mesenchyme,0.35,0.17,0.46,0.31,1.05,0.03


In [5]:
pd.crosstab(obs.celltype, obs.stage).loc[
    ["Anterior", "Posterior", "Osteogenic", "Early/unpatterned mesenchyme"]]

stage,E12.5,E13.5,E14.0,E14.5
celltype,,,,
Anterior,62,2543,2603,3009
Posterior,56,1075,1240,1250
Osteogenic,0,370,1206,324
Early/unpatterned mesenchyme,10036,2940,1972,1144


**Why no chondrocyte call.** *Sox9* and *Col2a1* are expressed in these early clusters,
which is what a relative (z-score) ranking keys on. *Acan* — a committed-cartilage marker —
is not.

In [6]:
acan = detection_fraction(adata, "Acan")
mes = obs.celltype.isin(["Early/unpatterned mesenchyme"])
early_clusters = obs.loc[mes, "leiden"].unique()
(acan.loc[early_clusters] * 100).round(2).sort_values(ascending=False).rename("% cells Acan+")

25    3.23
20    2.93
18    2.92
24    1.47
26    1.24
12    0.23
8     0.21
3     0.05
4     0.04
5     0.00
Name: % cells Acan+, dtype: float64

## 3. The pilot: RNA already predicts fate in "unpatterned" progenitors

Train on E14.5 anterior vs posterior, apply to the 10,036 E12.5 cells with no A/P marker
signature. Three controls are built into `fate_prediction` and the stage-axis removal.

In [7]:
p = cfg["priming"]
X = adata.obsm["X_pca"]
train = ((obs.stage == p["train_stage"]) & obs.celltype.isin(p["classes"])).to_numpy()
y = (obs.celltype[train] == p["classes"][0]).astype(int).to_numpy()
apply_to = ((obs.stage == p["apply_stage"]) & (obs.celltype == p["apply_celltype"])).to_numpy()

# control 1: remove the maturation axis, estimated within mesenchyme only
both = (obs.stage.isin([p["apply_stage"], p["train_stage"]])
        & obs.celltype.isin(p["stage_axis_celltypes"])).to_numpy()
X_r = remove_axis(X, stage_axis(X, (obs.stage == p["train_stage"]).to_numpy(), both, cfg["seed"]))

res = fate_prediction(X_r, train, y, apply_to, cfg)   # controls 2+3: balanced weights, permuted null
print(f"training AUROC: {res['auroc_mean']:.3f}  (n={res['n_train']:,})")
print(f"confident calls on {res['n_apply']:,} E12.5 progenitors:")
print(f"  real labels     {res['frac_confident']:.1%}")
print(f"  permuted labels {res['frac_confident_null']:.1%}")

training AUROC: 0.999  (n=4,259)
confident calls on 10,036 E12.5 progenitors:
  real labels     82.3%
  permuted labels 0.0%


What does skipping the controls do? This is the reason they exist.

In [8]:
naive = fate_prediction(X, train, y, apply_to, {**cfg, "priming": {**p, "class_weight": None}})
print(f"no stage-axis removal, no class balancing: {naive['frac_confident']:.1%} confident "
      f"(null {naive['frac_confident_null']:.1%})")
print(f"with all controls:                         {res['frac_confident']:.1%} confident "
      f"(null {res['frac_confident_null']:.1%})")

no stage-axis removal, no class balancing: 76.8% confident (null 12.1%)
with all controls:                         82.3% confident (null 0.0%)


## What this means

"Unpatterned" meant *no marker signature*, not *no information*: the transcriptome
already predicts eventual A/P fate for most E12.5 progenitors. So the original hypothesis —
chromatin is primed *before* transcription — has a false premise here.

The testable version is: **does chromatin add predictive power over RNA, at matched
feature count and sparsity?** That needs the consensus ATAC matrix (not yet built) and the
same `fate_prediction` call on LSI components.

**Caveat.** The classifier is trained at E14.5 and applied at E12.5. Some of the signal may
reflect lineage continuity rather than early fate commitment; the controls rule out
maturation and base-rate artefacts, not that.